# Taller 1 - Econometría II
**Estudiante:** María Perdomo  
**Tema:** Estimación por Máxima Verosimilitud (AR1 con ruido Laplace) y Métodos Numéricos (MA1 con Newton-Raphson)


## Punto 1: AR(1) con Ruido Blanco Laplaciano

### 1. Ecuaciones de Primer Orden (CPO)

Asumiendo que los errores siguen una distribución de Laplace con parámetro de localización 0 y escala $b$:
$$f(\epsilon_t) = \frac{1}{2b} \exp\left(-\frac{|\epsilon_t|}{b}\right)$$

#### Caso 1: Modelo sin intercepto ($y_t = \rho y_{t-1} + \epsilon_t$)
La función de log-verosimilitud condicional es:
$$\ln L(\rho, b) = -T \ln(2b) - \frac{1}{b} \sum_{t=1}^T |y_t - \rho y_{t-1}|$$

Derivando respecto a $\rho$ (usando la función signo $\text{sgn}(x)$ para la derivada del valor absoluto):
$$\frac{\partial \ln L}{\partial \rho} = \frac{1}{b} \sum_{t=1}^T y_{t-1} \text{sgn}(y_t - \rho y_{t-1}) = 0$$

#### Caso 2: Modelo con intercepto ($y_t = c + \rho y_{t-1} + \epsilon_t$)
La log-verosimilitud queda:
$$\ln L(c, \rho, b) = -T \ln(2b) - \frac{1}{b} \sum_{t=1}^T |y_t - c - \rho y_{t-1}|$$

Las CPO son:
1. $\frac{\partial \ln L}{\partial c} = \frac{1}{b} \sum_{t=1}^T \text{sgn}(y_t - c - \rho y_{t-1}) = 0$
2. $\frac{\partial \ln L}{\partial \rho} = \frac{1}{b} \sum_{t=1}^T y_{t-1} \text{sgn}(y_t - c - \rho y_{t-1}) = 0$

---

### 2. ¿Existe solución analítica cerrada?

**No**, ninguno de los dos casos tiene solución cerrada. 

*¿Por qué?* 
A diferencia de OLS donde derivamos errores al cuadrado $(y_t - \rho y_{t-1})^2$ y obtenemos ecuaciones lineales fáciles de despejar, aquí la derivada del valor absoluto introduce la función $\text{sgn}()$, que es escalonada y discontinua en cero. No hay forma algebraica de despejar $\rho$ o $c$. Se tiene que resolver numéricamente mediante optimización no lineal o programación lineal.

---

### 3. Laplace vs. Normal y conexión con Regresión LAD

* **Propiedades de la distribución:** La distribución de Laplace tiene colas mucho más pesadas que la Normal (curtosis de 6 vs 3). Esto significa que tolera de forma mucho más natural los valores atípicos (*outliers*) o shocks extremos sin sesgar los parámetros.
* **Conexión con LAD (Least Absolute Deviations):** Maximizar $\ln L$ bajo errores Laplace equivale a minimizar $\sum |\epsilon_t|$. Esto es exactamente la estimación LAD ($L_1$-norm), una alternativa robusta a Mínimos Cuadrados Ordinarios ($L_2$-norm).

---

### 4. Simulación y Estimación en Python


In [1]:
import numpy as np
from scipy.optimize import minimize

# Semilla aleatoria (fecha de nacimiento / id)
np.random.seed(2024)

# Parámetros verdaderos
T = 500
c_true = 0.8
rho_true = 0.65
b_true = 1.0

# Simulación del AR(1) con ruido Laplace
e = np.random.laplace(loc=0, scale=b_true, size=T)
y = np.zeros(T)
y[0] = c_true / (1 - rho_true) + e[0]

for t in range(1, T):
    y[t] = c_true + rho_true * y[t-1] + e[t]

# Función de log-verosimilitud negativa
def neg_log_like_laplace(params, y):
    c, rho, b = params
    if b <= 0 or abs(rho) >= 1:
        return 1e10
    e_t = y[1:] - c - rho * y[:-1]
    n = len(e_t)
    return n * np.log(2 * b) + np.sum(np.abs(e_t)) / b

# Estimación
res = minimize(neg_log_like_laplace, x0=[0.0, 0.0, 1.0], args=(y,), method='Nelder-Mead')
c_est, rho_est, b_est = res.x

print("=== RESULTADOS AR(1) LAPLACIANO ===")
print(f"c:   Verdadero = {c_true:.2f} | Estimado = {c_est:.3f}")
print(f"rho: Verdadero = {rho_true:.2f} | Estimado = {rho_est:.3f}")
print(f"b:   Verdadero = {b_true:.2f} | Estimado = {b_est:.3f}")


=== RESULTADOS AR(1) LAPLACIANO ===
c:   Verdadero = 0.80 | Estimado = 0.886
rho: Verdadero = 0.65 | Estimado = 0.606
b:   Verdadero = 1.00 | Estimado = 0.936


## Punto 2: Solución Numérica MA(1) Gaussiano con Newton-Raphson

### 1 y 2. Modelo MA(1) y Ecuaciones No Lineales

El modelo es $y_t = \mu + \epsilon_t + \theta \epsilon_{t-1}$ con $\epsilon_t \sim N(0, \sigma^2)$.

Como los shocks no se observan, los calculamos de forma recursiva asumiendo $\epsilon_0 = 0$:
$$\epsilon_t(\mu, \theta) = y_t - \mu - \theta \epsilon_{t-1}$$

Maximizamos la log-verosimilitud condicional, lo que equivale a minimizar la suma de residuos al cuadrado:
$$S(\mu, \theta) = \sum_{t=1}^T \epsilon_t(\mu, \theta)^2$$

El sistema de ecuaciones no lineales a resolver es $g(\mu, \theta) = \nabla S(\mu, \theta) = \mathbf{0}$:
$$g_1(\mu, \theta) = \sum_{t=1}^T \epsilon_t \frac{\partial \epsilon_t}{\partial \mu} = 0$$
$$g_2(\mu, \theta) = \sum_{t=1}^T \epsilon_t \frac{\partial \epsilon_t}{\partial \theta} = 0$$

donde las derivadas se obtienen recursivamente:
$$\frac{\partial \epsilon_t}{\partial \mu} = -1 - \theta \frac{\partial \epsilon_{t-1}}{\partial \mu}$$
$$\frac{\partial \epsilon_t}{\partial \theta} = -\epsilon_{t-1} - \theta \frac{\partial \epsilon_{t-1}}{\partial \theta}$$

---

### 3. Algoritmo de Newton-Raphson

Usamos la actualización vectorial:
$$\mathbf{\Theta}^{(k+1)} = \mathbf{\Theta}^{(k)} - \mathbf{J}^{-1}(\mathbf{\Theta}^{(k)}) \cdot g(\mathbf{\Theta}^{(k)})$$
donde $\mathbf{J}$ es la matriz Jacobiana del sistema $g$.


In [2]:
import numpy as np

# Simulación del proceso MA(1)
np.random.seed(2024)
N = 300
mu_true = 2.0
theta_true = 0.5
sigma_true = 1.0

eps = np.random.normal(0, sigma_true, N)
y_ma = np.zeros(N)
y_ma[0] = mu_true + eps[0]
for t in range(1, N):
    y_ma[t] = mu_true + eps[t] + theta_true * eps[t-1]

# Función para calcular residuos y derivadas
def eval_ma1(mu, theta, y_data):
    T_len = len(y_data)
    e_val = np.zeros(T_len)
    de_dmu = np.zeros(T_len)
    de_dtheta = np.zeros(T_len)
    
    for t in range(T_len):
        e_prev = e_val[t-1] if t > 0 else 0.0
        dmu_prev = de_dmu[t-1] if t > 0 else 0.0
        dth_prev = de_dtheta[t-1] if t > 0 else 0.0
        
        e_val[t] = y_data[t] - mu - theta * e_prev
        de_dmu[t] = -1.0 - theta * dmu_prev
        de_dtheta[t] = -e_prev - theta * dth_prev
        
    g1 = np.sum(e_val * de_dmu)
    g2 = np.sum(e_val * de_dtheta)
    return np.array([g1, g2]), e_val, de_dmu, de_dtheta

# Algoritmo Newton-Raphson con diferencias finitas para el Jacobiano
def newton_raphson_ma1(y_data, theta_init, tol=1e-8, max_iter=50):
    theta_curr = np.array(theta_init, dtype=float)
    h = 1e-6
    
    print(f"{'Iter':<5} | {'mu':<10} | {'theta':<10} | {'Norma Grad':<12}")
    print("-" * 45)
    
    for k in range(max_iter):
        g_curr, _, _, _ = eval_ma1(theta_curr[0], theta_curr[1], y_data)
        norm_g = np.linalg.norm(g_curr)
        
        print(f"{k:<5} | {theta_curr[0]:<10.5f} | {theta_curr[1]:<10.5f} | {norm_g:<12.4e}")
        
        if norm_g < tol:
            print(f"Convergencia lograda en la iteración {k}!")
            break
            
        # Matriz Jacobiana numérica de g
        J = np.zeros((2, 2))
        g_m1, _, _, _ = eval_ma1(theta_curr[0] + h, theta_curr[1], y_data)
        g_m2, _, _, _ = eval_ma1(theta_curr[0], theta_curr[1] + h, y_data)
        
        J[:, 0] = (g_m1 - g_curr) / h
        J[:, 1] = (g_m2 - g_curr) / h
        
        # Actualización
        step = np.linalg.solve(J, g_curr)
        theta_curr -= step
        
    return theta_curr

# Ejecución
init_guess = [np.mean(y_ma), 0.1]
mu_est, theta_est = newton_raphson_ma1(y_ma, init_guess)

print("=== COMPARACIÓN DE RESULTADOS MA(1) ===")
print(f"mu:    Verdadero = {mu_true:.2f} | Estimado = {mu_est:.3f}")
print(f"theta: Verdadero = {theta_true:.2f} | Estimado = {theta_est:.3f}")


Iter  | mu         | theta      | Norma Grad  
---------------------------------------------
0     | 1.93337    | 0.10000    | 9.6671e+01  
1     | 1.93005    | 0.53591    | 1.1174e+01  
2     | 1.93197    | 0.50875    | 6.2557e-01  
3     | 1.93190    | 0.50704    | 2.1603e-03  
4     | 1.93190    | 0.50703    | 3.0275e-08  
5     | 1.93190    | 0.50703    | 7.2288e-14  
Convergencia lograda en la iteración 5!
=== COMPARACIÓN DE RESULTADOS MA(1) ===
mu:    Verdadero = 2.00 | Estimado = 1.932
theta: Verdadero = 0.50 | Estimado = 0.507


### Notas finales
* La estimación del AR(1) con errores Laplace demuestra la efectividad de la regresión LAD cuando los errores tienen colas pesadas.
* Para el MA(1), la versión condicional de Newton-Raphson converge rápidamente (en pocas iteraciones) si el valor inicial de $\theta$ se encuentra dentro del rango de invertibilidad ($|\theta| < 1$).
